In [1]:
import numpy as np
from scipy.integrate import fixed_quad, quad
import os
import plotly.graph_objects as go
from scipy.special import j0, jv
import pandas as pd
import plotly.graph_objects as go

In [2]:
# Leitura do arquivo com separação por espaços
data_atlas = pd.read_csv(
    "../../../data/sigma_tot_2/ensemble_StRh_atlas.dat",
    delim_whitespace=True,
    header=None,
    nrows=70  # lê apenas as 70 primeiras linhas
)

x_atlas = data_atlas[0].to_numpy()
y_atlas = data_atlas[1].to_numpy()
y_error_atlas = data_atlas[2].to_numpy()

In [3]:
# === Global Configuration and Constants ===
start_sqrt_s = 1  # Global parameter controlling energy scale
b_0 = (33 - 6) / (12 * np.pi)  # β0 for nf=3
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0

sigma_tot_lst = []
sqrt_s_lst = []
error_lst = []

s0 = 1.0  # GeV^2

epsilon_atlas = 0.0729

model_params = {
    'atlas': {
        'pl':  {'mg': 0.412, 'a1': 1.652, 'a2': 1.479}
    }
}

epsilon_values = {
    'atlas': epsilon_atlas
}


In [4]:

# === Auxiliary Functions for Physical Model ===
def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)

def get_m2_function(mass_model):
    return m2_pl

def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, q, phi, mg, a1, a2, m2_func):
    q2 = q ** 2
    qk_cos = q * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2

    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)

    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, q, phi, mg, a1, a2, m2_func):
    q2 = q ** 2
    qk_cos = q * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2

    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)

    factor = q2 + 9 * abs(k ** 2 - q2 / 4)

    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)

    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def integrand(y, x, mg, a1, a2, m2_func):
    k = sqrt_s * x
    phi = 2 * np.pi * y
    jacobian = 2 * np.pi * sqrt_s

    return k * (T_1(k, 0.0, phi, mg, a1, a2, m2_func) - T_2(k, 0.0, phi, mg, a1, a2, m2_func)) * jacobian

def amp_calculation(diff_T, s, epsilon):
    alpha_pomeron = 1.0 + epsilon
    regge_factor = (s / s0) ** alpha_pomeron
    
    return 1j * 8.0 * regge_factor * diff_T

def sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323


In [5]:
amp_born_lst = []

sqrt_s_lst = []

# === Main Function ===
def main():
    global start_sqrt_s
    global sqrt_s

    max_sqrt_s = 13000
    step = 100
    n_points = 10000

    # Using only PL model with ATLAS
    mass_model = 'pl'
    ensemble = 'atlas'

    fig = go.Figure()

    sigma_tot_lst = []
    

    

    m2_func = get_m2_function(mass_model)
    params = model_params[ensemble][mass_model]
    mg, a1, a2 = params['mg'], params['a1'], params['a2']
    epsilon = epsilon_values[ensemble]

    sqrt_s = start_sqrt_s
    while sqrt_s <= max_sqrt_s:
        def inner_integral(x):
            return fixed_quad(
                lambda y: integrand(y, x, mg, a1, a2, m2_func),
                0, 1,
                n=n_points
            )[0]

        integral_value = fixed_quad(
            inner_integral,
            0, 1,
            n=n_points
        )[0]

        diff_T = integral_value
        s = sqrt_s * sqrt_s

        amp_value = amp_calculation(diff_T, s, epsilon)
        sigma_tot_value = sigma_tot(amp_value, s)

        sigma_tot_lst.append(sigma_tot_value)
        sqrt_s_lst.append(sqrt_s)
        amp_born_lst.append(amp_value)

        sqrt_s += step

    # Add PL model trace
    fig.add_trace(go.Scatter(
        x=sqrt_s_lst,
        y=sigma_tot_lst,
        mode='lines+markers',
        line=dict(
            color='blue',
            width=2
        ),
        marker=dict(
            size=4
        ),
        name='PL Model (ATLAS)'
    ))

    # Add ATLAS data
    fig.add_trace(go.Scatter(
        x=x_atlas,
        y=y_atlas,
        mode='markers',
        marker=dict(
            color='black',
            size=6,
            symbol='square'
        ),
        error_y=dict(
            type='data',
            array=y_error_atlas,
            visible=True
        ),
        name='ATLAS Data'
    ))

    # Configure layout
    fig.update_layout(
        title='Sigma Tot vs. sqrt(s) - PL Model with ATLAS Data',
        xaxis=dict(
            title='sqrt(s) [GeV]',
            type='log',
        ),
        yaxis=dict(
            title='Sigma Tot [mb]',
        ),
        showlegend=True,
        legend=dict(
            title='Model/Data'
        ),
        plot_bgcolor='white',
        hovermode='x unified'
    )
    
    fig.update_xaxes(gridcolor='lightgray')
    fig.update_yaxes(gridcolor='lightgray')

    # fig.show(renderer="browser")
    # fig.write_html("results/sigma_tot/sigma_tot_pl_atlas.html")
    # fig.write_image("results/sigma_tot/sigma_tot_pl_atlas.pdf", width=1200, height=600)


In [6]:
if __name__ == "__main__":
    main()

In [7]:
lst_s = []
for key, value in enumerate(sqrt_s_lst):
    lst_s.append(value ** 2)
    # print(f"sqrt(s) = {value:.2f} GeV, s = {lst_s[key]:.2f} GeV^2")

# print(amp_born_lst)


In [8]:
print(amp_born_lst)

[63.115184501271365j, 1347190.7176672097j, 5898675.860122347j, 14030218.73631267j, 25964720.860653378j, 41866654.51598634j, 61867890.78110346j, 86079194.04572104j, 114596439.07204364j, 147504383.44482678j, 184879140.67330098j, 226789894.8495602j, 273300141.18166864j, 324468614.13086754j, 380350000.93986464j, 440995502.6309437j, 506453283.48836726j, 576768837.040938j, 651985288.2303321j, 732143645.9378079j, 817283016.2906829j, 907440784.5550761j, 1002652771.559333j, 1102953369.2413766j, 1208375658.9160113j, 1318951515.11109j, 1434711697.2539096j, 1555685931.0526915j, 1681902981.0784702j, 1813390715.785762j, 1950176165.9985693j, 2092285577.7186654j, 2239744459.9761634j, 2392577628.331045j, 2550809244.54304j, 2714462852.8520713j, 2883561413.249028j, 3058127332.0645657j, 3238182490.159889j, 3423748268.966669j, 3614845574.5919394j, 3811494860.1772504j, 4013716146.6786423j, 4221529042.2143974j, 4434952760.110781j, 4654006135.761345j, 4878707642.402832j, 5109075405.899548j, 5345127218.61863j,

In [ ]:
n = 100

q_upper_limit = 0.2
b_upper_limit = 10


def inner_integral(b, s, amp):
    integrand = lambda q: q * j0(b*q)
    result, _ = quad(integrand, 0, q_upper_limit, limit=200)  # limite de subdivisões
    return result* (1/s) *amp


def outer_integrand(b, s, amp):
    inner_result = inner_integral(b, s, amp)
    return 1j * s * b * (1 - np.exp(1j*inner_result))


def outer_real(b, s, amp):
    return np.real(outer_integrand(b, s, amp))

def outer_imag(b, s, amp):
    return np.imag(outer_integrand(b, s, amp))



def compute_double_integral(s, amp):
    real_part, _ = quad(lambda b: outer_real(b, s, amp), 0, b_upper_limit, limit=500)
    imag_part, _ = quad(lambda b: outer_imag(b, s, amp), 0, b_upper_limit, limit=500)
    return real_part + 1j * imag_part


lst_amp_eik = []

for s_val, amp_born_val in zip(lst_s, amp_born_lst):
    amp_eik_val = compute_double_integral(s_val, amp_born_val)
    # print(f"Numerical result: {result1} for s = {s_val} and amp born = {amp_born_val}")
    lst_amp_eik.append(amp_eik_val)

print(lst_amp_eik)


TypeError: outer_real() missing 1 required positional argument: 'amp'

In [ ]:
def sigma_tot_eik(s, amp):
    return (4*np.pi)/s * amp.imag * 0.389379323

lst_sigma_tot_eik = []

for amp_eik_val, s_val in zip(lst_amp_eik, lst_s):
    sigma_tot_eik_val = sigma_tot_eik(s_val, amp_eik_val)
    lst_sigma_tot_eik.append(sigma_tot_eik_val)

In [ ]:
lst_imag_val_amp_eik = []
for values in lst_amp_eik:
    img_part = values.imag
    lst_imag_val_amp_eik.append(img_part)



In [ ]:
def create_iterative_graph(fig, x_data, y_data, title:str, x_axis_name:str, y_axis_name:str):

    fig.add_trace(go.Scatter(
    x = x_data,
    y = y_data,
    mode='lines+markers')
)

    fig.update_layout(
        title=title,
        xaxis=dict(
            title = x_axis_name,
            type='log',
        ),
        yaxis=dict(
            title= y_axis_name,
            # range=[80, 120]
        ),
        showlegend=False,
        plot_bgcolor='white',
        hovermode='x unified'
    )
        
    fig.update_xaxes(gridcolor='lightgray')
    fig.update_yaxes(gridcolor='lightgray')



In [ ]:
fig_amp_eik = go.Figure()

create_iterative_graph(fig_amp_eik, sqrt_s_lst, lst_imag_val_amp_eik, 'Amp eikonal vs sqrt', 'sqrt s [GeV]', 'Im(Amp_eikonal)')

# os.makedirs("../../../results/amp_eikonal", exist_ok=True)
# file_name = 'amp_eikonal.pdf'
# save_path = os.path.join("../../../results/amp_eikonal", file_name)

fig_amp_eik.show(renderer = 'browser')

# fig.write_image(save_path, width=1200, height=600)

In [ ]:
fig_sigma_eik = go.Figure()

create_iterative_graph(fig_sigma_eik, sqrt_s_lst, lst_sigma_tot_eik, 'sigma eik vs sqrt', 'sqrt s [GeV]', 'sigma eik [mb]')

# os.makedirs("../../../results/amp_eikonal", exist_ok=True)
# file_name = 'amp_eikonal.pdf'
# save_path = os.path.join("../../../results/amp_eikonal", file_name)

fig_sigma_eik.show(renderer = 'browser')

# fig.write_image(save_path, width=1200, height=600)